In [5]:
#!pip3 install autodistill-efficient-yolo-world
#!pip install roboflow
#!pip install --upgrade inference

In [2]:
from autodistill_efficient_yolo_world import EfficientYOLOWorld
from autodistill.detection import CaptionOntology
import cv2
import supervision as sv

In [3]:
from pathlib import Path
from llama_index.core import SimpleDirectoryReader
import os
import numpy as np
from tqdm import tqdm

In [4]:
# define an ontology to map class names to our EfficientYOLOWorld prompt
# the ontology dictionary has the format {caption: class}
# where caption is the prompt sent to the base model, and class is the label that will
# be saved for that caption in the generated annotations
# then, load the model
base_model = EfficientYOLOWorld(ontology=CaptionOntology({"colorful circle part": "colorful circle part"}))

Creating inference sessions


Specified provider 'CUDAExecutionProvider' is not in available provider names.Available providers: 'CPUExecutionProvider'
Specified provider 'OpenVINOExecutionProvider' is not in available provider names.Available providers: 'CPUExecutionProvider'


CLIP model loaded in 0.99 seconds


/root/.autodistill/efficient_sam_s_gpu.jit: No such file or directory


ValueError: The provided filename /root/.autodistill/efficient_sam_s_gpu.jit does not exist

In [ ]:
# load as image documents
image_documents = SimpleDirectoryReader("/home/data/ISIC2019/raw", required_exts=[".jpeg", ".jpg", ".png"], recursive=True, filename_as_id=True).load_data()


In [ ]:
def metadata_str2dict(string):
    # Split the string into lines
    lines = string.strip().split("\n")
    
    # Create a dictionary to store the key-value pairs
    file_info_dict = {}
    
    # Parse each line and add key-value pairs to the dictionary
    for line in lines:
        key, value = line.split(": ", 1)
        file_info_dict[key] = value
    
    return(file_info_dict)

In [ ]:
image_patch = []
image_no_patch = []

for image in image_documents:
    metadata = metadata_str2dict(image.get_metadata_str())
    image_path = metadata['file_path']

    folder_name = os.path.basename(os.path.dirname(image_path))
    
    if folder_name == "patch":
        metadata['patch'] = True
        image_patch.append({'image': image, 'meta': metadata})


    elif folder_name == "no_patch":
        metadata['patch'] = False
        image_no_patch.append({'image': image, 'meta': metadata})


    else:
        print("The jpg is not in either the patch or no_patch folder.")

In [ ]:
num_imgs = 100
num_imgs_per_class = int(np.ceil(num_imgs/2))

# combine images with and without patch
images = []
images.extend(image_patch[:num_imgs_per_class])
images.extend(image_no_patch[:num_imgs_per_class])

# output file
data_dict = {}
#data_dict['model_path'] = model_path
#data_dict['tokenizer_path'] = tokenizer_path
#data_dict['chat_template'] = chat_template
data_dict['num_imgs'] = num_imgs


In [ ]:
def run_inference(images, num_imgs, question1, question2=None, data_dict=None):
    correct_classified = 0
    false_classified = 0
    more_than_one_word = 0
    num_yes_patch = 0
    num_no_no_patch = 0
    num_yes = 0
    num_no = 0

    output_dict = None
    if data_dict is not None:
        if question2 is None:
            data_dict['prompt']= question1
        else:
            data_dict['question1']= question1
            data_dict['question2']= question2
        output_dict = {}


    print("-----------------------------------------\n")
    
    print(question1)
    if question2 is not None:
        print('\n', question2)
    
    for i, image_dict in enumerate(tqdm(images[:num_imgs])):
        image = image_dict['image']
        meta = image_dict['meta']
        
        ## display image
        #imageUrl = meta['file_path']
        #image = Image.open(imageUrl).convert("RGB")
        #plt.figure(figsize=(8, 3))
        #plt.imshow(image)
    
        # image
        image_path = meta['file_path']


        # predict on an image
        result = base_model.predict(image_path, confidence=0.0025)
        
        image = cv2.imread(image_path)
        
        mask_annotator = sv.MaskAnnotator()
        annotated_frame = mask_annotator.annotate(
        	scene=image.copy(),
        	detections=result,
        )

        sv.plot_image(annotated_frame)

        #base_model.label("./context_images", extension=".jpeg")
        
        
        if response in {"yes", "yes."}:
            #print("yes")
            num_yes += 1
            if meta['patch']:
                correct_classified += 1
                num_yes_patch += 1
                #print("correct")
            else:
                false_classified += 1
                #print("false")
    
        elif response in {"no", "no."}:
            #print("no")
            
            num_no += 1
            if meta['patch']:
                false_classified += 1
                #print("false")
            else:
                correct_classified += 1
                num_no_no_patch +=1
                #print("correct")
        else:
            #print("No yes or no!")
            more_than_one_word += 1

        # save data
        if output_dict is not None:
            img_output_dict = {}
            
            if question2 is None:
                img_output_dict['answer'] =  state["answer"]
            else:
                img_output_dict['img_meta'] = image_dict['meta']
                img_output_dict['explanation'] = state["explanation"]
                img_output_dict['is_patch'] = state["output"]
            output_dict[f'img_{i}'] = img_output_dict


    acc = correct_classified/(num_imgs - more_than_one_word)
    
    print("\nAccuracy: ",acc)
    print("correct classified: ", correct_classified,"\nfalse classified: ", false_classified, 
          "\nNumber yes: ", num_yes, "\nNumber no: ", num_no, "\nNeither yes or no: ", more_than_one_word,
          "\nNumber True positives: ", num_yes_patch, "\nNumber True negatives: ", num_no_no_patch,
          "\nnum images: ", num_imgs)
    print("\n-----------------------------------------\n\n")

    # save data
    if data_dict is not None:
        data_dict['accuracy'] = acc
        data_dict['correct_classified'] = correct_classified
        data_dict['false_classified'] = false_classified
        data_dict['num_yes'] = num_yes
        data_dict['num_no'] = num_no
        data_dict['more_than_one_word'] = more_than_one_word
        data_dict['true_positive'] = num_yes_patch
        data_dict['true_negative'] = num_no_no_patch
        data_dict['outputs']=output_dict

        # Mkdir if not existent
        path="/home/outputfiles"
        os.makedirs(path, exist_ok=True)
        
        # Generate a timestamp
        timestamp = datetime.now().strftime("%Y%m%dT%H%M%S")

        # Create a filename with the timestamp
        filename = f"{path}/llava_output_{str(correct_classified)}_{timestamp}.json"
        
        with open(filename, 'w', encoding='utf-8') as f:
            json.dump(data_dict, f, indent=4)